<div style="background:#A6192E;color:#fff;padding:22px 28px;border-radius:14px;font-family:'Barlow Condensed','Arial Narrow',Arial,sans-serif">
<div style="letter-spacing:.2em;text-transform:uppercase;font-size:13px;opacity:.9">Python para Finanzas y Análisis de Datos · Sesión 1</div>
<div style="font-size:34px;font-weight:300;text-transform:uppercase;margin-top:6px"><span style="background:#fff;color:#A6192E;border-radius:10px;padding:2px 12px;margin-right:14px;font-weight:500">1.2</span>Valoración de opciones: Black-Scholes</div>
</div>

**Objetivo.** valorar una opción europea con la fórmula cerrada, verificarla con Monte Carlo y obtener griegas y volatilidad implícita.

**Lo que vamos a hacer**

- Implementar Black-Scholes.
- Comprobar el precio simulando trayectorias (lo visto en 1.1).
- Calcular delta, gamma y vega por diferencias finitas.
- Invertir la fórmula para hallar la volatilidad implícita.

In [ ]:
import sys, pathlib, subprocess, warnings
warnings.filterwarnings("ignore")

# Localiza la carpeta del curso (en Google Colab clona el repositorio)
root = pathlib.Path.cwd().resolve()
while root != root.parent and not (root / "common").exists():
    root = root.parent
if not (root / "common").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Ulysess824/Python-for-Finance"], check=False)
    root = pathlib.Path("Python-for-Finance/python-finance-course").resolve()
sys.path.insert(0, str(root))
# En Colab, ejecuta antes:  !pip install -q yfinance arch

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. Fórmula cerrada

$$d_1=\frac{\ln(S/K)+(r+\sigma^2/2)T}{\sigma\sqrt T},\quad d_2=d_1-\sigma\sqrt T,\quad C=S\,N(d_1)-Ke^{-rT}N(d_2)$$

In [ ]:
from scipy.stats import norm
S0, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20

> ### ✏️ Ejercicio 1 — bs_price
> Implementa `bs_price(S, K, T, r, sigma, kind)` para call y put. Debe dar ≈ 10,45 con los datos de arriba.

In [ ]:
def bs_price(S, K, T, r, sigma, kind="call"):
    # TODO
    pass

In [ ]:
print(f"bs_price = {bs_price(S0, K, T, r, sigma, 'call'):.2f}   (esperado: 10.45)")

## 2. Verificación con Monte Carlo

Bajo riesgo-neutral, $S_T=S_0\exp\big((r-\sigma^2/2)T+\sigma\sqrt T\,z\big)$. El precio es el promedio descontado del pago.

> ### ✏️ Ejercicio 2 — mc_price
> Devuelve `(precio, error_estándar)` simulando `n` valores de $S_T$.

In [ ]:
def mc_price(S, K, T, r, sigma, kind="call", n=200_000, seed=1):
    rng = np.random.default_rng(seed)
    # TODO
    return None, None

In [ ]:
p_mc, se = mc_price(S0, K, T, r, sigma)
print(f"Black-Scholes: {bs_price(S0, K, T, r, sigma):.4f}")
print(f"Monte Carlo  : {p_mc:.4f} ± {1.96 * se:.4f}  (IC 95%)")

## 3. Griegas por diferencias finitas

In [ ]:
h = 0.01 * S0
base = bs_price(S0, K, T, r, sigma)
up, dn = bs_price(S0 + h, K, T, r, sigma), bs_price(S0 - h, K, T, r, sigma)
greeks = pd.Series({
    "Delta": (up - dn) / (2 * h),
    "Gamma": (up - 2 * base + dn) / h**2,
    "Vega (por 1% de vol)": (bs_price(S0, K, T, r, sigma + .01) - bs_price(S0, K, T, r, sigma - .01)) / 2,
})
display(greeks.round(4))

S_grid = np.linspace(60, 140, 100)
fig, ax = plt.subplots()
ax.plot(S_grid, [bs_price(s, K, T, r, sigma) for s in S_grid], label="Valor de la call hoy")
ax.plot(S_grid, np.maximum(S_grid - K, 0), color="#5f5f5f", ls="--", label="Pago al vencimiento")
ax.set_xlabel("Precio del subyacente"); ax.set_title("Call europea"); ax.legend(); plt.show()

## 4. Volatilidad implícita

> ### ✏️ Ejercicio 3 — implied_vol
> Una call ATM cotiza a 12,0. Encuentra la volatilidad que reproduce ese precio con `scipy.optimize.brentq`.

In [ ]:
from scipy.optimize import brentq

def implied_vol(price, S, K, T, r, kind="call"):
    # TODO
    pass

In [ ]:
print(f"Vol implícita de la call a 12.0: {implied_vol(12.0, S0, K, T, r):.2%}")

## Para llevarse

- La fórmula cerrada y Monte Carlo coinciden; Monte Carlo sirve cuando no hay fórmula.
- Delta mide la sensibilidad al precio, gamma la curvatura y vega la sensibilidad a la volatilidad.
- La volatilidad implícita es la que el mercado «paga»; se obtiene invirtiendo la fórmula numéricamente.

---
*Excel para mirar, Python para repetir.*